# Demo 01 — Introduction to Machine Learning

**Machine Learning with scikit-learn — Session 01**

โน้ตบุ๊ครันตามสไลด์คาบ 1 — ทุก cell คือโค้ดตัวเดียวกับที่อยู่บนสไลด์ เรียงตามหมายเลขสไลด์
วิธีใช้: เปิดสไลด์คู่กัน แล้วรัน `Shift+Enter` ตามที่อาจารย์ไปถึงหน้าไหน

## Slide 12 — Meet the iris dataset (1936!)

ข้อมูลดอกไม้ 150 ดอก × 4 คุณลักษณะ (เซนติเมตร) — เป้าหมาย: ทำนายสายพันธุ์ (0/1/2)

In [1]:
from sklearn.datasets import load_iris
import pandas as pd

X, y = load_iris(return_X_y=True)
print(X.shape, y.shape)   # (150, 4) และ (150,)
pd.DataFrame(X, columns=load_iris().feature_names).head()

(150, 4) (150,)


,sepal length (cm),sepal width (cm),petal length (cm),petal width (cm)
0,5.1,3.5,1.4,0.2
1,4.9,3.0,1.4,0.2
2,4.7,3.2,1.3,0.2
3,4.6,3.1,1.5,0.2
4,5.0,3.6,1.4,0.2


## Slide 13 — Your first model, five lines

นี่คือ "สวัสดีชาว Machine Learning" — จำลำดับให้ขึ้นใจ: **แบ่งข้อมูล → สร้างโมเดล → fit → ตรวจคะแนน**

In [2]:
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42)          # ① hold out 30%

model = KNeighborsClassifier(n_neighbors=3)        # ② create
model.fit(X_train, y_train)                        # ③ learn
print(model.score(X_test, y_test))                 # ④ judge

1.0


ได้ **1.0** — ทายถูก 100% บนดอกไม้ 45 ดอกที่โมเดล **ไม่เคยเห็นมาก่อน**

อย่าเคยชินกับเลขสวยขนาดนี้นะครับ iris เป็นข้อมูลฝึกที่ "ง่ายแบบใจดี" — ข้อมูลจริง (Titanic คาบหน้า) จะโหดกว่านี้มาก

## Slide 14 — Never grade a student on questions it has seen

train/test split คือหัวใจของการประเมินโมเดล:
- `fit` เรียนรู้จาก training set อย่างเดียว
- `score` วัดผลบน test set ที่โมเดลไม่เคยเห็น = "ข้อสอบจริง"
- `random_state=42` ทำให้ผลลัพธ์ทำซ้ำได้ทุกเครื่อง

In [3]:
print("train size:", len(X_train), "| test size:", len(X_test))
print("train accuracy:", round(model.score(X_train, y_train), 3))
print("test  accuracy:", round(model.score(X_test, y_test), 3))

train size: 105 | test size: 45
train accuracy: 0.943
test  accuracy: 1.0


## Slide 15 — Look at what it got wrong

โมเดลก็พลาดได้ — นัก data science ที่ดีต้อง "อ่าน" ความผิดพลาด ไม่ใช่แค่ดูคะแนน
เราขยับ k ไปที่ 50 เพื่อให้เห็นว่าโมเดลพลาดหน้าตาเป็นอย่างไร

In [4]:
model50 = KNeighborsClassifier(n_neighbors=50).fit(X_train, y_train)
y_pred = model50.predict(X_test)

compare = pd.DataFrame({"actual": y_test, "predicted": y_pred})
compare[compare.actual != compare.predicted]     # แสดงเฉพาะแถวที่ทายผิด

,actual,predicted
21,2,1
40,2,1


พลาด 2 จาก 45 ดอก — และพลาดเป็นทางเดียวกันหมด: ทาย virginica (2) ว่าเป็น versicolor (1)
เพราะสองสายพันธุ์นี้ซ้อนทับกันในเชิงขนาดดอก มนุษย์ก็งงนะครับ

## Slide 16 — KNN in 30 seconds

กลไกข้างในของ KNN สั้นมาก: หาเพื่อนบ้าน k ตัวที่ใกล้ที่สุด → ให้เพื่อนบ้านโหวต

In [5]:
# ดู probability ที่โมเดลคิด (ไม่ใช่แค่คำตอบสุดท้าย)
model.predict_proba(X_test[:3]).round(3)

array([[0., 1., 0.],
       [1., 0., 0.],
       [0., 0., 1.]])

## Slide 17 — Turn one knob, watch the score move

`n_neighbors` (k) คือ **hyperparameter** — ค่าที่ "เรา" เลือกก่อนเทรน
ลองบิด k แล้วดูคะแนน: ต่ำไป? สูงไป? เหมือนกันหมด?

In [6]:
for k in [1, 3, 5, 15, 50]:
    m = KNeighborsClassifier(n_neighbors=k)
    m.fit(X_train, y_train)
    print(f"k={k:2d}  accuracy={m.score(X_test, y_test):.3f}")

k= 1  accuracy=1.000
k= 3  accuracy=1.000
k= 5  accuracy=1.000
k=15  accuracy=1.000
k=50  accuracy=0.956


iris ง่ายมากจนแทบทุก k ก็สมบูรณ์แบบ — จนกระทั่ง k=50 ห่างเกินไป คะแนนตกลง (underfitting)

ในข้อมูลที่ยากกว่านี้ การเลือก k จะสำคัญมาก — และวิธีเลือกอย่างถูกต้องชื่อ **cross-validation** เราจะเรียนในคาบ 4

## Slide 19 — Prompting AI for ML code that actually works

สูตร prompt ที่ดีสำหรับงาน ML: **CONTEXT + GOAL + CONSTRAINT + VERIFY** — ลองเล่นจริงกับ cell นี้
(ใน Colab ลองกดปุ่ม Generate code / ถาม AI ที่คุณชอบก็ได้)

```
I have a pandas DataFrame `df` with columns tenure(float), contract(object),
monthly_charges(float), churn(0/1 int). Using scikit-learn only:
1) build a ColumnTransformer that imputes+one-hot-encodes contract
   and scales the numeric columns,
2) chain it with LogisticRegression in a Pipeline,
3) evaluate with 5-fold stratified CV — mean ROC-AUC.
Random-state 42. Do NOT touch a holdout test set.
```

แล้วอย่าลืมขั้นสุดท้าย: **VERIFY** — เช็กว่า API มีจริง พารามิเตอร์ไม่ตาย และไม่มี leakage ซ่อนอยู่

---
✅ **จบ Demo 01** — คุณเพิ่งเทรนโมเดล ML ตัวแรกของชีวิต อ่านความผิดพลาด และบิด hyperparameter เป็นแล้ว

**ต่อไป:** ไปทำ **Lab 01** ใน `excercises/01.ml_intro_exercise.ipynb` — โจทย์ภาษาไทย ลองทำเองก่อนดูเฉลยนะครับ